<!-- Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. -->

# MJX 08 — Playground PPO: Ball-in-Cup

A second MuJoCo Playground control task: **`BallInCup`** (DM Control Suite). A
planar actuator must swing a ball on a string into a cup — a sparse-reward,
dynamic task that is noticeably harder than balancing a cartpole but still
learns reliably in a few minutes on the AMD GPU.

Same stable recipe as MJX 07: pure-JAX physics (`impl="jax"`), `num_envs=256`,
and highest matmul precision.

In [ ]:
import os, functools
os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jp
if not hasattr(jax, "device_put_replicated"):
    def _dpr(x, devices=None):
        n = len(devices) if devices is not None else jax.local_device_count()
        return jax.tree_util.tree_map(
            lambda a: jax.device_put(jp.broadcast_to(jp.asarray(a)[None], (n,) + jp.asarray(a).shape)), x)
    jax.device_put_replicated = _dpr
jax.config.update("jax_default_matmul_precision", "highest")

import numpy as np
import matplotlib.pyplot as plt
import imageio
from mujoco_playground import registry, wrapper
from mujoco_playground.config import dm_control_suite_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from IPython.display import Video

print("JAX devices:", jax.devices())

In [ ]:
ENV_NAME = "BallInCup"
env = registry.load(ENV_NAME, config_overrides={"impl": "jax"})

cfg = dm_control_suite_params.brax_ppo_config(ENV_NAME)
ppo_params = cfg.to_dict()
net_cfg = ppo_params.pop("network_factory", None)
ppo_params.update(
    num_timesteps=4_000_000,
    num_envs=256,
    batch_size=256,
    num_minibatches=8,
    num_evals=10,
)
print({k: ppo_params[k] for k in ["num_timesteps", "num_envs", "batch_size", "episode_length"]})

In [ ]:
progress = []
def progress_fn(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    progress.append((int(step), r))
    print(f"step {int(step):>9}  eval reward {r:8.1f}")

train_fn = functools.partial(ppo.train, **ppo_params)
if net_cfg:
    train_fn = functools.partial(train_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))

make_inference_fn, params, _ = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    seed=0,
)
print("training done")

In [ ]:
steps, rewards = zip(*progress)
plt.figure(figsize=(7, 3))
plt.plot(steps, rewards, marker="o")
plt.xlabel("environment steps"); plt.ylabel("eval episode reward")
plt.title(f"PPO learning curve ({ENV_NAME})"); plt.grid(True); plt.show()

In [ ]:
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

rng = jax.random.PRNGKey(1)
state = reset(rng)
trajectory = [state]
for _ in range(250):
    rng, k = jax.random.split(rng)
    action, _ = inference(state.obs, k)
    state = step(state, action)
    trajectory.append(state)

frames = np.asarray(env.render(trajectory, height=240, width=320))
out = "output/videos/mjx08_ball_in_cup.mp4"
imageio.mimsave(out, list(frames), fps=30)
print("saved", frames.shape[0], "frames ->", out)

In [ ]:
Video(url="output/videos/mjx08_ball_in_cup.mp4")